# 08 — Connected execution workflow

Use **reports** mode to generate existing deep-learning and biological graphics, **train** mode to additionally train all architectures and run corrected feature ablation, or **full** mode to process raw GDC inputs first. The separate outputs are generated by notebooks 07 and 10.

In [ ]:
from pathlib import Path
import nbformat
from nbclient import NotebookClient

def find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p/"config"/"genes.json").exists(): return p
    raise FileNotFoundError("Cannot locate project root")
ROOT=find_root()
NB=ROOT/"notebooks"
EXEC=NB/"executed"
EXEC.mkdir(exist_ok=True)
# "reports" generates biology and deep learning figures without retraining.
# "train" runs existing architecture notebooks and corrected ablation before reporting.
# "full" also rebuilds raw GDC processed tables (raw data must be present locally).
MODE="reports"
KERNEL_NAME="python3"  # Change to your active virtualenv kernelspec if needed.
experiment_plan=[
    "02_Experiment_1_Standard_MLP.ipynb",
    "03_Experiment_2_Dropout_Comparison.ipynb",
    "04_Experiment_3_BatchNorm_Comparison.ipynb",
    "05_Experiment_4_Residual_MLP.ipynb",
    "06_Experiment_5_MultiBranch_MultiTask.ipynb",
]
reports_plan=["07_Deep_Learning_Results.ipynb",
              "10_Biological_Results.ipynb",
              "11_Export_Results_Tables.ipynb"]
if MODE=="reports":
    plan=reports_plan
elif MODE=="train":
    plan=experiment_plan+["09_Feature_Group_Ablation.ipynb"]+reports_plan
elif MODE=="full":
    plan=["01_Data_Processing_Pipeline.ipynb"]+experiment_plan+["09_Feature_Group_Ablation.ipynb"]+reports_plan
else:
    raise ValueError("Choose MODE = 'reports', 'train', or 'full'")
if MODE!="full" and not (ROOT/"data"/"processed"/"modeling_table.csv").exists():
    raise FileNotFoundError("Missing data/processed/modeling_table.csv")
print("Execution mode:",MODE)
print("\n".join(plan))

In [ ]:
for name in plan:
    path=NB/name
    print("EXECUTING",name,flush=True)
    with open(path,encoding="utf-8") as f:
        notebook=nbformat.read(f,as_version=4)
    client=NotebookClient(
        notebook,kernel_name=KERNEL_NAME,timeout=7200,allow_errors=False,
        resources={"metadata":{"path":str(ROOT)}}
    )
    try:
        client.execute()
    finally:
        # Keep executed notebook even when a cell raises an exception.
        with (EXEC/name).open("w",encoding="utf-8") as f:
            nbformat.write(notebook,f)
    print("PASSED",name)
print("All selected notebooks completed")